In [7]:
from services.clock_analysis import (
    compute_clock,
    compute_normalized_share
)

import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("data/processed/cleaned.csv")

clock_data = compute_clock(df)

normalized_share = compute_normalized_share(
    clock_data
)

C:\Users\ashan\AppData\Local\Temp\ipykernel_13096\626399232.py:9: DtypeWarning: Columns (0: offline) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/processed/cleaned.csv")


In [32]:
df["ts"] = pd.to_datetime(df["ts"])
df["hour"] = df["ts"].dt.hour

# Minutes listened by each user at each hour
hourly = (
    df.groupby(["hour", "user"])["minutes"]
    .sum()
    .reset_index()
)

# Total listening time per user
user_totals = (
    df.groupby("user")["minutes"]
    .sum()
)

# Normalize within each user
hourly["share"] = hourly.apply(
    lambda r:
        r["minutes"] /
        user_totals[r["user"]]
        * 100,
    axis=1
)

pivot = (
    hourly
    .pivot(
        index="hour",
        columns="user",
        values="share"
    )
    .fillna(0)
)

pivot = pivot.reindex(
    range(24),
    fill_value=0
)

In [ ]:
import plotly.graph_objects as go

colors = {
    "Ashanti": "#4F6BFF",
    "Gabi": "#FF4FB8",
    "Maribel": "#FF8A3D"
}

fig = go.Figure()

for user in pivot.columns:

    values = pivot[user]

    max_value = values.max()

    sizes = [
        18 if v == max_value else 8
        for v in values
    ]

    fig.add_trace(
        go.Scatter(
            x=pivot.index,
            y=values,

            mode="lines+markers",

            line=dict(
                color=colors[user],
                width=2
            ),

            marker=dict(
                color=colors[user],
                size=sizes
            ),

            opacity=0.9,

            showlegend=False,
            name=user
        )
    )

    # Label at end of line

    last_x = values.dropna().index[-1]
    last_y = values.dropna().iloc[-1]

    label_offsets = {
        "Ashanti": 0.10,
        "Gabi": 0.25,
        "Maribel": -0.15
    }

    fig.add_annotation(
        x=last_x,
        y=last_y + label_offsets[user],

        text=f"<b>{user}</b>",

        showarrow=False,

        xshift=40,

        font=dict(
            color=colors[user],
            size=15
        )
    )

fig.update_layout(
    title="Distribution of Each User's Listening Across the Day",

    xaxis=dict(
        title="Hour of Day",
        dtick=1,
        range=[-0.5, 23.5],
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        title="% of User's Total Listening",
        showgrid=False,
        zeroline=False
    ),

    margin=dict(
        l=60,
        r=140,
        t=60,
        b=60
    ),

    template="plotly_dark",
    height=550
)

import plotly.io as pio

pio.renderers.default = "browser"

fig.show()

In [30]:
df["ts"] = pd.to_datetime(df["ts"])
df["year"] = df["ts"].dt.year

yearly = (
    df.groupby(["year", "user"])["minutes"]
    .sum()
    .reset_index()
)

# convert minutes -> hours
yearly["hours"] = yearly["minutes"] / 60

colors = {
    "Ashanti": "#4F6BFF",
    "Gabi": "#FF4FB8",
    "Maribel": "#FF8A3D"
}

fig = go.Figure()

for user in yearly["user"].unique():

    user_df = yearly[
        yearly["user"] == user
    ].sort_values("year")

    fig.add_trace(
        go.Scatter(
            x=user_df["year"],
            y=user_df["hours"],

            mode="lines+markers",

            line=dict(
                color=colors[user],
                width=4
            ),

            marker=dict(
                size=10,
                color=colors[user]
            ),

            name=user
        )
    )

fig.update_layout(
    title="Listening Hours by Year",

    xaxis=dict(
        title="Year",
        dtick=1,
        showgrid=False
    ),

    yaxis=dict(
        title="Listening Hours",
        showgrid=False
    ),

    template="plotly_dark",
    height=550
)

fig.show()

In [29]:
import pandas as pd
import plotly.graph_objects as go

# -------------------------
# Prepare data
# -------------------------

df["ts"] = pd.to_datetime(df["ts"])
df["year"] = df["ts"].dt.year

# Remove Maribel before 2023

df = df[
    ~(
        (df["user"] == "Maribel")
        &
        (df["year"] < 2023)
    )
]

# -------------------------
# Listening per year
# -------------------------

yearly = (
    df.groupby(["year", "user"])["minutes"]
    .sum()
    .reset_index()
)

pivot = (
    yearly.pivot(
        index="year",
        columns="user",
        values="minutes"
    )
    / 60
)

# -------------------------
# Cumulative hours
# -------------------------

cumulative = pd.DataFrame(index=pivot.index)

for user in pivot.columns:

    cumulative[user] = pivot[user].cumsum()

    first_year = pivot[user].first_valid_index()

    cumulative.loc[
        cumulative.index < first_year,
        user
    ] = None

# -------------------------
# Colors
# -------------------------

colors = {
    "Ashanti": "#4F6BFF",
    "Gabi": "#FF4FB8",
    "Maribel": "#FF8A3D"
}

# -------------------------
# Plot
# -------------------------

fig = go.Figure()

for user in cumulative.columns:

    user_data = cumulative[user].dropna()

    fig.add_trace(
        go.Scatter(
            x=user_data.index,
            y=user_data.values,

            mode="lines+markers",

            line=dict(
                color=colors[user],
                width=3
            ),

            marker=dict(
                size=8,
                color=colors[user]
            ),

            showlegend=False,
            name=user
        )
    )

    fig.add_annotation(
        x=user_data.index[-1],
        y=user_data.iloc[-1],

        text=f"<b>{user}</b>",

        showarrow=False,

        xshift=45,

        font=dict(
            size=16,
            color=colors[user]
        )
    )

# -------------------------
# Layout
# -------------------------

fig.update_layout(

    title="Cumulative Listening Hours Over Time",

    template="plotly_dark",

    height=600,

    xaxis=dict(
        title="Year",
        dtick=1,
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        title="Cumulative Listening Hours",
        showgrid=False,
        zeroline=False,

        tickvals=[0,1000,2000,3000,4000,5000,6000,7000],
        ticktext=["0","1k","2k","3k","4k","5k","6k","7k"]
    ),

    margin=dict(
        l=80,
        r=120,
        t=80,
        b=60
    )
)

fig.show()